# 05 · CUB70 and CBM — does a positive concept require visible pixels?

Professor Ramaswamy's question is evaluated on real CUB photographs with CUB70 part masks.

For concept `j`, the original processed CUB label is `c_j`. The model produces raw value `z_j` and probability `c_pred_j`. The mask gives `visible_part(j)`.

We restrict the main test to rows with `c_j=1`, then compare:

`c_pred_j | visible_part(j)=1` versus `c_pred_j | visible_part(j)=0`.

If an originally-positive concept falls when its named part is not visible, the output is responsive to visible evidence. If it remains high, information elsewhere in the image is sufficient to produce that concept.

This is an observational occlusion test, not a deletion counterfactual: naturally visible and occluded images may differ in pose, species, and background. It can demonstrate a grounding problem but cannot by itself identify species as the unique source.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CURATED = Path(os.environ["CURATED_DATA"])
CWD = Path.cwd(); REPO = CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0, str(REPO/"analysis"))
sys.path.insert(0, str(REPO/"data"/"cub70"))
from occlusion import (attach_visibility, z_by_visibility, grounding_violation_rate,
    quartile_grounding, within_species_visibility_effect,
    visibility_specificity_control)
from relabel_cub_with_cub70 import coarse_visibility
try:
    from plotting import set_paper_style, PALETTE; set_paper_style(); CBM_C=PALETTE["CBM"]
except Exception: CBM_C="#0072B2"

def need(p, how):
    p=Path(p)
    if not p.exists(): print(f"[pending] {p}\n  produce it: {how}")
    return p.exists()

vis_path=CURATED/"cub70_visibility.parquet"
VIS=None
if need(vis_path, "bash data/cub70/prepare_all.sh"):
    VIS=coarse_visibility(pd.read_parquet(vis_path), threshold=0.001).rename(columns={"coarse":"part"})
    print(f"masks: {VIS.image_name.nunique()} images, parts={sorted(VIS.part.unique())}")


## 1 · Full-CUB-trained CBM on the masked CUB70 images

First use the CBM trained on all 200 CUB classes. Joining its test predictions to CUB70 masks automatically restricts the analysis to masked images.

For every part, print:

- `prob_mean = mean(c_pred_j | c_j=1, visible)`;
- `z_median = median(z_j | c_j=1, visible)`;
- `violation_rate = P(c_pred_j≥0.5 | c_j=1, visible=0)`.

Here “violation” means the model still reports an originally-positive concept when its named part mask is absent. Because the image was not actively edited, it is evidence of nonlocal sufficiency, not proof that species alone caused the answer.


In [ ]:
SEED=1
full_path=CURATED/"cub70_eval"/f"cub-cbm-s{SEED}.parquet"
J_FULL=None
if VIS is not None and need(full_path, "CONFIGS='cub-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh"):
    E=pd.read_parquet(full_path)
    J_FULL=attach_visibility(E[E.part!=""], VIS)
    Z=z_by_visibility(J_FULL); display(Z.round(3))
    V=grounding_violation_rate(J_FULL); display(V.round(3))
    P=Z.pivot(index="part",columns="visible",values="prob_mean").rename(columns={False:"occluded",True:"visible"})
    fig,ax=plt.subplots(figsize=(7,3.5)); x=np.arange(len(P)); w=.38
    ax.bar(x-w/2,P.get("occluded"),w,label="mask absent",color="#CC79A7")
    ax.bar(x+w/2,P.get("visible"),w,label="mask visible",color=CBM_C)
    ax.set_xticks(x); ax.set_xticklabels(P.index,rotation=30,ha="right")
    ax.set_ylim(0,1); ax.set_ylabel("mean c_pred_j for original c_j=1")
    ax.set_title("Full-CUB CBM: concept output versus part visibility"); ax.legend(); plt.show()


## 2 · Does the amount of visible area matter?

A binary mask-present flag may hide a graded relationship. Among rows with `c_j=1`, divide `area_frac` into within-part quartiles and compute

`mean(c_pred_j | visibility quartile)`.

A rising curve means the concept output tracks how much of the part is visible. A flat high curve means the same answer is produced with little or much visible part evidence. Counts are required because some part/visibility groups may be small.


In [ ]:
if J_FULL is not None:
    Q=quartile_grounding(J_FULL); display(Q.round(4))
    fig,ax=plt.subplots(figsize=(7,3.6))
    for part,d in Q.groupby("part"):
        ax.plot(d.qbin,d.prob_mean,"o-",label=part)
    ax.set_xticks(sorted(Q.qbin.unique())); ax.set_xlabel("within-part mask-area quartile")
    ax.set_ylabel("mean c_pred_j for original c_j=1"); ax.set_ylim(0,1)
    ax.set_title("Full-CUB CBM: response across visible-area quartiles"); ax.legend(ncol=2); plt.show()


## 3 · What does visibility relabeling change?

CUB70 masks exist only for test images. We may therefore ask how many evaluation labels change from `c_j=1` to visible-absence `c_j^vis=0`, but we cannot claim to have retrained the model with these masks.

For mapped attribute `j`:

`c_j^vis = c_j × visible_part(j)`.

The flip table measures label/mask disagreement by part. It answers whether segmentation adds information beyond the species-level label. It is not an original-versus-relabeled training comparison.


In [ ]:
diag_path=CURATED/"cub70_relabel_diagnostics.parquet"
if need(diag_path, "python data/cub70/relabel_cub_with_cub70.py"):
    D=pd.read_parquet(diag_path)
    S=(D.groupby("part").agg(considered=("flipped","size"),flipped=("flipped","sum")))
    S["flip_rate_all"]=S.flipped/S.considered
    pos=D[D.original_label==1].groupby("part").agg(original_positive=("flipped","size"),flipped=("flipped","sum"))
    pos["flip_rate_given_positive"]=pos.flipped/pos.original_positive
    R=S.join(pos[["original_positive","flip_rate_given_positive"]]); display(R.round(3))
    fig,ax=plt.subplots(figsize=(7,3.3)); ax.bar(R.index,R.flip_rate_given_positive,color=CBM_C)
    ax.set_ylim(0,1); ax.set_ylabel("P(mask absent | original c_j=1)")
    ax.set_title("CUB70: positive labels contradicted by visibility mask"); plt.xticks(rotation=30,ha="right"); plt.show()


## 4 · Repeat after training a genuine 70-class CBM

The full-CUB and CUB70-trained models are evaluated on the same masked images with the same original labels. The only intended model-side change is that the second model was trained to distinguish 70 rather than 200 species.

Compare `violation_rate` and the visible/occluded probability gap. This addresses the professor's “same test for the CUB70 model” request. It does not constitute visibility-aware retraining because training masks do not exist.


In [ ]:
cub70_path=CURATED/"cub70_eval"/f"cub70-cbm-s{SEED}.parquet"
if VIS is not None and need(cub70_path, "SEEDS='1' bash train/cbm_cub70.sh; then CONFIGS='cub70-cbm' bash analysis/cub70_prepare_analysis.sh"):
    E70=pd.read_parquet(cub70_path); J70=attach_visibility(E70[E70.part!=""],VIS)
    rows=[]
    for name,J in [("full-CUB trained",J_FULL),("CUB70 trained",J70)]:
        if J is None: continue
        v=grounding_violation_rate(J)
        rows += [dict(model=name,part=r.part,n_occluded=r.n_occluded,violation_rate=r.violation_rate) for r in v.itertuples()]
    C=pd.DataFrame(rows); display(C.round(3))
    H=C.pivot(index="part",columns="model",values="violation_rate")
    H.plot.bar(figsize=(8,3.6),ylim=(0,1),color=[CBM_C,"#009E73"])
    plt.ylabel("P(c_pred_j≥0.5 | original c_j=1, mask absent)")
    plt.title("Same masked images: full-CUB versus CUB70-trained CBM"); plt.xticks(rotation=30,ha="right"); plt.show()


## 5 · Species-matched control

<!-- SPECIES-MATCHED CONTROL -->
The previous plots can accidentally compare one set of species when a part is visible with a different set when it is hidden. That matters because standard CUB concept labels are species-majority labels.

For each fixed triple `(species y, concept j, part)` that contains both visible and occluded photographs, calculate

`Δ_yj = mean(c_pred_j | visible, y, j) − mean(c_pred_j | occluded, y, j)`.

Then average `Δ_yj` within each part. Positive `Δ_yj` means visibility still matters after holding species and the exact concept fixed. Near-zero `Δ_yj` means the naive gap was largely between species. This removes the species main effect, but pose and viewpoint remain observational confounds.


In [ ]:
if J_FULL is not None:
    MATCHED=within_species_visibility_effect(J_FULL)
    display(MATCHED.round(4))
    fig,ax=plt.subplots(figsize=(7,3.4))
    ax.bar(MATCHED.part,MATCHED.visible_minus_occluded,color=CBM_C)
    ax.axhline(0,color="black",lw=1)
    ax.set_ylabel("within-(species, concept) Δ c_pred_j")
    ax.set_title("CBM: visibility effect after holding species fixed")
    plt.xticks(rotation=30,ha="right");plt.show()


## 6 · Negative-label specificity control

The grounding claim concerns concepts with `c_j=1`. We also calculate the visible-minus-occluded probability difference when `c_j=0`. If positive and negative labels move together, visibility may be acting as a general pose or image-quality signal instead of evidence specific to the named positive concept.


In [ ]:
if J_FULL is not None:
    CONTROL=visibility_specificity_control(J_FULL)
    display(CONTROL.round(4))
    H=CONTROL.pivot(index="part",columns="gt_label",values="visible_minus_occluded")
    H=H.rename(columns={0:"original c_j=0",1:"original c_j=1"})
    H.plot.bar(figsize=(8,3.5),color=["#999999",CBM_C])
    plt.axhline(0,color="black",lw=1)
    plt.ylabel("mean c_pred_j(visible) − mean c_pred_j(occluded)")
    plt.title("Is the visibility response specific to positive concepts?")
    plt.xticks(rotation=30,ha="right");plt.show()


## 7 · Seed support and final decision

A single seed is exploratory. Load every exported `cub-cbm-s*.parquet`, repeat the species-matched effect, and print the number of seeds. The notebook may describe a pattern from one seed, but it may call it dependable only when the same sign appears across independent seeds.


In [ ]:
seed_rows=[]
if VIS is not None:
    for path in sorted((CURATED/"cub70_eval").glob("cub-cbm-s*.parquet")):
        seed=int(path.stem.rsplit("-s",1)[1])
        J=attach_visibility(pd.read_parquet(path).query("part != ''"),VIS)
        for r in within_species_visibility_effect(J).itertuples():
            seed_rows.append(dict(seed=seed,part=r.part,n_matched_groups=r.n_matched_groups,
                                  visible_minus_occluded=r.visible_minus_occluded))
if seed_rows:
    SEEDS=pd.DataFrame(seed_rows);display(SEEDS.round(4))
    SUMMARY=(SEEDS.groupby("part").visible_minus_occluded
             .agg(["mean","std","count","min","max"]).reset_index())
    display(SUMMARY.round(4))
    print("Plain reading: positive means the named part's visibility still changes c_pred_j "
          "after species and concept are held fixed. count is the number of model seeds.")
else:
    print("[pending] export at least one full-CUB CBM: CONFIGS='cub-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh")


## Conclusion boundary

This notebook can establish whether `c_pred_j` remains positive without a visible named part, whether probability changes with mask area, and whether that effect survives a within-species comparison. It cannot prove that species recognition is the unique source of an occluded answer, because CUB70 contains natural photographs rather than paired edits. The causal source test remains the controlled FunnyBirds swap/deletion experiment in notebook 02.
